# Post-Processing

Now that we have the JSON files of extracted text, let's see how we can edit them into individual event text files for further processing

In [3]:
import json
import re
import csv
from pathlib import Path
from statistics import mean, median

# ── Directories ──────────────────────────────────────────────
RAW_DIR   = Path("gemini_output")          # input: raw OCR JSONs
CLEAN_DIR = Path("transliteration_json")   # output: cleaned per-page JSONs
CSV_FILE  = Path("events.csv")             # output: merged events

CLEAN_DIR.mkdir(exist_ok=True)

# ── Patterns ──────────────────────────────────────────────────
VEKAYI_PATTERN  = re.compile(r"VEK[AÂĀ]Y[İI]", re.IGNORECASE)

SUSPECT_PATTERNS = [
    re.compile(r'm[aâ]dde',       re.IGNORECASE),
    re.compile(r'n[aâ]zm',        re.IGNORECASE),
    re.compile(r'n[aâ]z[iıî]m',   re.IGNORECASE),
    re.compile(r'm[iîı]sra',      re.IGNORECASE),
    re.compile(r':$'),  # subheading ends with colon
    re.compile(r'\.$'), # subheading ends with period
    re.compile(r',$'), # subheading ends with comma
    re.compile(r';$'), # subheading ends with semicolon
]


## Extract the response

In [25]:
def parse_raw_response(raw):
    """Strip markdown fences and parse JSON. Returns (parsed_dict, error_str)."""
    if raw is None:
        return None, "raw_response is None"

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()

    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e:
        return None, str(e)

## Calculate the subheading length distributions

In [26]:
def compute_normal_subheading_median(json_files):
    """
    Calculate the median subheading length from 'normal' events across all pages.
    Normal = has both subheading and body, subheading is shorter than body, not a year marker.
    """
    lengths = []

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            wrapper = json.load(f)

        parsed, err = parse_raw_response(wrapper.get("raw_response"))
        if err:
            continue

        for ev in parsed.get("events", []):
            if not isinstance(ev, dict):
                continue
            sub  = ev.get("subheading") or ""
            body = ev.get("body")
            body_is_empty = body is None or body == "null" or body == ""

            if (sub
                    and not body_is_empty
                    and not VEKAYI_PATTERN.search(sub)
                    and len(sub) < len(body)):
                lengths.append(len(sub))

    return median(lengths)

## Fixes

There are 4 major issues that needs to be addressed in the post-processing

1. Body key missing

Handles cases where the model put all text into subheading and omitted the body key entirely. The subheading text is appended to the previous event's body and the misformed event is removed

2. Subheading is suspicious

Handles the cases where the subheading contains some terms that the model often mistook for a new subheadings, like nazm, misra, tarihidir: etc. The text from the subheading is appended to the FRONT of the body of the same event followed by a line break and the subheading is set to null

For example (page 327)
```python
{
    "subheading": "Beşinci mâdde",
    "body": "Hotin Kalʻası ve kazâsında vâkiʻ reʻâyâ-yı Rûsiyyelü ile sulh tanzîm olununcaya dek ber-veche-i emânet Nemçelü tarafında kalup, Rûsiyyelü'ye vechen mine'l-vücûh iʻânet ü imdâd eylemeyeler."
},
{
    "subheading": "Altıncı madde",
    "body": null
}
```

becomes

```python
{
    "subheading": null,
    "body": "Beşinci mâdde\n\nHotin Kalʻası ve kazâsında vâkiʻ reʻâyâ-yı Rûsiyyelü ile sulh tanzîm olununcaya dek ber-veche-i emânet Nemçelü tarafında kalup, Rûsiyyelü'ye vechen mine'l-vücûh iʻânet ü imdâd eylemeyeler."
},
{
    "subheading": null,
    "body": "Altıncı madde\n\n"
}
```

This results in these events looking like consecutive null subheading events, which we then merge all the way up in our usual workflow
   
3. Body is null and subheading is not a year marker

Handles the cases where there is no body to the text but the subheading is not a year.

4. Subheading is significantly longer than the body

This is a variation of the third case but the main difference is that there is some amount of text in the body. What makes a subheading significantly longer is if it is both longer than the body and the median length of a normal subheading. A normal case is where the subheading and body both exist and the subheading is shorter than the body.

Both of these cases will be handled like the suspicious words


For example page 398
```python
{
    "subheading": null,
    "body": "taʻyîn olunup, firâra mecâl olmadığından istîmân ile teslîm-i nefs-i bed-sigâl eylediklerinde, hayyen Âsitâne-i saʻâdet'e irsâl ve vürûdlarında izâle vü in'idâmlarıyla gāret-zede-i dest-i tetâvül olan fukarâ müreffehü'l-hâl oldular. Merkūmun bir oğlu ki, menbit-i sûdan hâsıl ve bir şahs-ı zâlim ü kātil idi, Kütahya Kalʻası'nda mahbûs olduğu mahsûs olmuşidi. Anın dahi cezâsı tertîb ü inkılâ‘-1 bîh-i vücûdlarıyla kâr-hâne-i zulm ü te'addîleri tahrîb olundu."    
},   
{
    "subheading": "İspanya Elçisi bu esnâda devleti tarafından matlûb olduğunu mübeyyen ibrâz-ı mektûb edüp, oğlunu Maslahat-güzâr sûretinde tevkīf niyâzını tekrîr ü taz‘îf eylediği Rikâb-ı müstetâb'a ‘arz [151b] u ifade olundukda mazhar-ı müsâʻade olunup, Bâb-ı ‘âlî'ye ihzâr ve ferve-i semmûr ile mâye-dâr-ı iftihâr kılındığından gayri, mukaddemâ İspanyalu ile vuku‘ bulan musâlaha hıdmetinde bulunmak hasebiyle mesfûra Istabl-ı hâs'dan bir re's-i müzeyyen esb ihsân olunup, mutayyeben memleketi tarafına revân oldu.",     
    "body": null    
},    
{
    "subheading": "Tefâsîl-i ahvâli zîr ü bâlâda serd ü beyân olunan İskenderiyyeli Kara Mahmûd'un i‘dâmiyçün Rumeli Vâlisi Vezîr Ebû Bekir Paşa devlet ‘askeri ve etrâfda vâki‘ Arnabud paşalar ile me'mûr kılındı.",    
    "body": null    
},    
{
    "subheading": "Gümrükçü Hasan Ağa'nın ‘uhde-i iltizâmında olan Gümrük Muhâsebesi ve ber-vech-i emânet idâresine me'mûr olduğu zecriyye mukātaʻasından hafî vü celî tahsîl eylediği mebâliğ, Defter Emîni bulunan Hakkı Beyefendi maʻrifetiyle teftîş olunup, zimmetinde mütehakkık olan emvâl cânib-i mîrîye teslîm ve Gümrük Emâneti'ne Ordu Kassâb-başısı Hazînedâr-zâde Mustafâ Bey işbu Cumâdelâhıre'de takdîm ve zecriyye mukātaʻasının idâresi dahi muʻayyen meʻâş ile Galata Voyvodası Halîl Efendi'ye tenbîh ü îsâ ve huzûr-ı Sadrı's-sudûr'da hil'âtleri iksâ olundu."
    "body": "Gümrükçü-yi sâbıkın zecriyyeden vâfir mâl cem' eylediği 'aklen ve naklen zâhir ve müsâdere ihtimâli zihne mütebâdir iken, Şehriyâr-ı merhamet-kâr ‘afv u safh ile"
}
```
becomes

```python
{
    "subheading": null,
    "body": "taʻyîn olunup, firâra mecâl olmadığından istîmân ile teslîm-i nefs-i bed-sigâl eylediklerinde, hayyen Âsitâne-i saʻâdet'e irsâl ve vürûdlarında izâle vü in'idâmlarıyla gāret-zede-i dest-i tetâvül olan fukarâ müreffehü'l-hâl oldular. Merkūmun bir oğlu ki, menbit-i sûdan hâsıl ve bir şahs-ı zâlim ü kātil idi, Kütahya Kalʻası'nda mahbûs olduğu mahsûs olmuşidi. Anın dahi cezâsı tertîb ü inkılâ‘-1 bîh-i vücûdlarıyla kâr-hâne-i zulm ü te'addîleri tahrîb olundu."    
},   
{
    "subheading": null,     
    "body": "İspanya Elçisi bu esnâda devleti tarafından matlûb olduğunu mübeyyen ibrâz-ı mektûb edüp, oğlunu Maslahat-güzâr sûretinde tevkīf niyâzını tekrîr ü taz‘îf eylediği Rikâb-ı müstetâb'a ‘arz [151b] u ifade olundukda mazhar-ı müsâʻade olunup, Bâb-ı ‘âlî'ye ihzâr ve ferve-i semmûr ile mâye-dâr-ı iftihâr kılındığından gayri, mukaddemâ İspanyalu ile vuku‘ bulan musâlaha hıdmetinde bulunmak hasebiyle mesfûra Istabl-ı hâs'dan bir re's-i müzeyyen esb ihsân olunup, mutayyeben memleketi tarafına revân oldu."    
},    
{
    "subheading": null,    
    "body": "Tefâsîl-i ahvâli zîr ü bâlâda serd ü beyân olunan İskenderiyyeli Kara Mahmûd'un i‘dâmiyçün Rumeli Vâlisi Vezîr Ebû Bekir Paşa devlet ‘askeri ve etrâfda vâki‘ Arnabud paşalar ile me'mûr kılındı."    
},    
{
    "subheading": null,
    "body": "Gümrükçü Hasan Ağa'nın ‘uhde-i iltizâmında olan Gümrük Muhâsebesi ve ber-vech-i emânet idâresine me'mûr olduğu zecriyye mukātaʻasından hafî vü celî tahsîl eylediği mebâliğ, Defter Emîni bulunan Hakkı Beyefendi maʻrifetiyle teftîş olunup, zimmetinde mütehakkık olan emvâl cânib-i mîrîye teslîm ve Gümrük Emâneti'ne Ordu Kassâb-başısı Hazînedâr-zâde Mustafâ Bey işbu Cumâdelâhıre'de takdîm ve zecriyye mukātaʻasının idâresi dahi muʻayyen meʻâş ile Galata Voyvodası Halîl Efendi'ye tenbîh ü îsâ ve huzûr-ı Sadrı's-sudûr'da hil'âtleri iksâ olundu.\n\nGümrükçü-yi sâbıkın zecriyyeden vâfir mâl cem' eylediği 'aklen ve naklen zâhir ve müsâdere ihtimâli zihne mütebâdir iken, Şehriyâr-ı merhamet-kâr ‘afv u safh ile"
}
```

In this particular work, we will lose one event that I know of through this merger: event 2 on page 327
```python
{
    "subheading": "İcmâl-i musâlaha-i Nemçe",
    "body": null
},
```

This is supposed to be the beginning of a treatise with Austria but all the clauses below it are split into individual events. I will fix this manually and hope that since we edited the few-shot examples after realizing that this was happening with the model, we will not encounter more examples of this.

This way of editing the subheading and leaving it null allows me to make changes to the transliteration_json files before merging the events

In [27]:
def fix_events(events, normal_subheading_median):
    """
    Fix all known event problems in a single pass.

    Case 1 — body key missing entirely:
        Merge subheading text up into previous event's body. Event is dropped.
        (Schema is broken, must fix immediately.)

    Cases 2, 3, 4 — restructure in place:
        Move subheading to front of body with \\n\\n separator, set subheading to null.
        Event stays as its own entry for inspection before cross-page merging.

        Case 2 — suspect keyword in subheading (madde, nazm, mısra, colon-ending)
        Case 3 — null body, not a year marker
        Case 4 — subheading longer than both body AND normal subheading median
    """
    if not isinstance(events, list):
        return events, []

    cleaned    = []
    fixed_idxs = []

    for idx, ev in enumerate(events, start=1):
        if not isinstance(ev, dict):
            cleaned.append(ev)
            continue

        sub  = ev.get("subheading") or ""
        body = ev.get("body")
        body_is_empty = body is None or body == "null" or body == ""

        # ── Case 1: body key missing entirely ────────────────
        if "body" not in ev:
            if cleaned and isinstance(cleaned[-1], dict):
                prev_body = cleaned[-1].get("body") or ""
                cleaned[-1]["body"] = f"{prev_body} {sub}".strip()
            fixed_idxs.append(("case1", idx))
            continue  # drop this event

        # ── Skip year markers ─────────────────────────────────
        if sub and VEKAYI_PATTERN.search(sub) and body_is_empty:
            cleaned.append(ev)
            continue

        # ── Case 2: suspect keyword in subheading ────────────
        is_suspect_keyword = bool(sub) and any(
            pat.search(sub) for pat in SUSPECT_PATTERNS
        )

        # ── Case 3: null body, not a year marker ─────────────
        is_null_body = bool(sub) and body_is_empty

        # ── Case 4: subheading longer than body AND median ───
        is_long_subheading = (
            bool(sub)
            and not body_is_empty
            and len(sub) > len(body or "")
            and len(sub) > normal_subheading_median
        )

        if is_suspect_keyword or is_null_body or is_long_subheading:
            # Restructure in place: subheading → front of body, subheading → null
            existing_body = "" if body_is_empty else (body or "")
            ev["body"] = f"{sub}\n\n{existing_body}".strip()
            ev["subheading"] = None
            case = ("case2" if is_suspect_keyword
                    else "case3" if is_null_body
                    else "case4")
            fixed_idxs.append((case, idx))

        cleaned.append(ev)

    return cleaned, fixed_idxs

In [28]:
# Compute once before the loop
raw_files = sorted(RAW_DIR.glob("page_*.json"), key=lambda p: int(p.stem.split("_")[1]))
parse_errors   = []
fix_log  = []

normal_median = compute_normal_subheading_median(raw_files)
print(f"Normal subheading median length: {normal_median} chars")

for jf in raw_files:
    with open(jf, "r", encoding="utf-8") as f:
        wrapper = json.load(f)

    # ── Parse inner JSON ──────────────────────────────────────
    parsed, err = parse_raw_response(wrapper.get("raw_response"))
    if err:
        parse_errors.append({"file": jf.name, "error": err})
        continue

    events = parsed.get("events", [])

    # ── Apply fixes ───────────────────────────────────────────

    events, fixed = fix_events(events, normal_median)
    if fixed:
        fix_log.append({
            "file":   jf.name,
            "fixed":  fixed  # list of (case, event_idx) tuples
        })

    # ── Save clean page ───────────────────────────────────────
    clean = {
        "page":   parsed.get("page"),
        "events": events
    }
    out_path = CLEAN_DIR / jf.name
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(clean, f, ensure_ascii=False, indent=2)

# # ── Summary ───────────────────────────────────────────────────
# print(f"✅ Processed {len(raw_files)} pages → {CLEAN_DIR}/")

# if parse_errors:
#     print(f"\n❌ Parse errors ({len(parse_errors)}):")
#     for e in parse_errors:
#         print(f"  - {e['file']}: {e['error']}")
# else:
#     print("✅ No parse errors")

# if fix_log:
#     print(f"\n🔧 Events fixed ({sum(len(x['fixed']) for x in fix_log)} events across {len(fix_log)} files):")
#     for item in fix_log:
#         print(f"  - {item['file']}: events {item['fixed']}")

# ── Summary ───────────────────────────────────────────────────
print(f"✅ Processed {len(raw_files)} pages → {CLEAN_DIR}/")

if parse_errors:
    print(f"\n❌ Parse errors ({len(parse_errors)}):")
    for e in parse_errors:
        print(f"  - {e['file']}: {e['error']}")
else:
    print("✅ No parse errors")

# Group fixes by case type
case_labels = {
    "case1": "Malformed events (body key missing)",
    "case2": "Suspect keyword in subheading",
    "case3": "Null body, non-year",
    "case4": "Subheading longer than body and median",
}

for case_key, case_label in case_labels.items():
    # Collect files and event indices for this case
    case_files = []
    for item in fix_log:
        idxs = [idx for (c, idx) in item["fixed"] if c == case_key]
        if idxs:
            case_files.append({"file": item["file"], "idxs": idxs})

    if case_files:
        total = sum(len(x["idxs"]) for x in case_files)
        print(f"\n🔧 {case_label} ({total} events across {len(case_files)} files):")
        for x in case_files:
            print(f"  - {x['file']}: events {x['idxs']}")

Normal subheading median length: 36 chars
✅ Processed 387 pages → transliteration_json/

❌ Parse errors (3):
  - page_154.json: Expecting value: line 1 column 1 (char 0)
  - page_192.json: Expecting value: line 1 column 1 (char 0)
  - page_216.json: Expecting value: line 1 column 1 (char 0)

🔧 Malformed events (body key missing) (3 events across 3 files):
  - page_399.json: events [2]
  - page_409.json: events [3]
  - page_511.json: events [2]

🔧 Suspect keyword in subheading (38 events across 16 files):
  - page_144.json: events [2]
  - page_145.json: events [1, 2, 3]
  - page_202.json: events [2]
  - page_278.json: events [2, 3, 4, 5]
  - page_312.json: events [2]
  - page_322.json: events [3, 4]
  - page_323.json: events [2]
  - page_327.json: events [3, 4, 5, 6, 7, 8]
  - page_328.json: events [2, 3, 4, 5, 6, 7]
  - page_329.json: events [2, 3]
  - page_356.json: events [2, 3]
  - page_398.json: events [2, 3, 4]
  - page_413.json: events [2]
  - page_424.json: events [2, 3, 4]
  - 

```python
Normal subheading median length: 36 chars
✅ Processed 387 pages → transliteration_json/

❌ Parse errors (3):
  - page_154.json: Expecting value: line 1 column 1 (char 0)
  - page_192.json: Expecting value: line 1 column 1 (char 0)
  - page_216.json: Expecting value: line 1 column 1 (char 0)

🔧 Malformed events (body key missing) (3 events across 3 files):
  - page_399.json: events [2]
  - page_409.json: events [3]
  - page_511.json: events [2]

🔧 Suspect keyword in subheading (38 events across 16 files):
  - page_144.json: events [2]
  - page_145.json: events [1, 2, 3]
  - page_202.json: events [2]
  - page_278.json: events [2, 3, 4, 5]
  - page_312.json: events [2]
  - page_322.json: events [3, 4]
  - page_323.json: events [2]
  - page_327.json: events [3, 4, 5, 6, 7, 8]
  - page_328.json: events [2, 3, 4, 5, 6, 7]
  - page_329.json: events [2, 3]
  - page_356.json: events [2, 3]
  - page_398.json: events [2, 3, 4]
  - page_413.json: events [2]
  - page_424.json: events [2, 3, 4]
  - page_425.json: events [2]
  - page_488.json: events [2]

🔧 Null body, non-year (2 events across 2 files):
  - page_327.json: events [2]
  - page_376.json: events [2]
```

This is pretty clean now!

Parse errors were expected because they were the few shot examples! We need to manually insert them. This will happen only for a few books where we used examples from them for our few-shots.

I also realized that I need to edit a few files:
page 139 last event
page 278, where the last event is not correctly parsed 
page 327 (see above)

ALSO IF THE BODY STARTS WITH LOWERCASE FLAG FOR THIS 
OR START WITH ' ETC BUT THEN CONTINUES WITH LOWERCASE INSTEAD OF UPPERCASE

## Merge continuations + year tracking

Walk through all cleaned pages in order. When event 1 on a page has `subheading == None`, it is a continuation of the previous event — append its body. When a VEKĀYİ year marker is encountered, update `current_year` and carry it forward to all subsequent events.

As a reminder here is what the extracted JSON looks like. These are two back to back pages:

```json
{
  "page": 147,
  "events": [
    {
      "subheading": "Nakl-i hazret-i Mehd-i ʻulyâ ez-Serây-ı ʻatîk be-Serây-ı cedîd-i sultânî",
      "body": "İklîletü'l-muhsanât, Seyyidetü'l-muhadderât, sadef-i dürr-i şeh-vâr-ı Devlet, evc-i âfitâb-ı Şehriyârî vü Saltanat hazretlerinin Serây-ı ʻatîk'den Serây-ı cedîd'e nakl ü hareketleri kānûn-ı kadîm ve de’b-i müstedîm olduğuna binâ’en, huzûrları muʻtâd olan erkân-ı Devlet dâhil ü hâric-i Serây-ı ʻatîk'de hâzır oldukları hâlde müşârun ileyha hazretleri gerdûne-i ʻismet-nümûnelerine süvâr ve tertîb-i dil-firîb, vâlâ-yı hûş-i resm ü ʻacîb ile Serây-ı cedîd'e bahşende-i şeref ü iʻtibâr olup, Şehriyâr-ı sütûde-etvâr, dâme mâ-dâme'l-felekü'd-devvâr \"el-Cennetü tahte akdâmi'l-ümmehât\" mefhûmu üzere dâʻiye-i tahsîl-i merzât ü mesûbât ile resm-i istikbâli kemâ-hüve hakkuhû icrâ ve müşârun ileyhâ hazretlerini Harem-serây-ı ʻismetlerine iskân ü îvâ buyurdular."
    },
    {
      "subheading": "Tevcih-i Kazâ’-i Havâss-ı refîʻa ve Nasb-ı Muhâsebe-i Haremeyn",
      "body": "İkinci Altmışlı Müderrisi olan ʻİzzet Beyefendi'nin kıdem-i intisâbı [8b] hasebiyle bir nevʻ ikrâm ile mükerrem olması hâtır-güzâr-ı Pâdişâh-ı dil-âgâh olmağla Mîr-i mûmâ ileyhe ‘avâtıf-ı ‘aliyyeden Havâss-ı refîʻa Mevleviyyeti muvakkaten tevcîh ve beyne'n-nâs kadrî terfîʻ vü tenvîh olundu. Kahveci-başı-yı Sânî olan ʻOsmân Efendi'ye dahi Yazıcı-yı esbak el-Hâc Ahmed Efendi ʻazlinden Haremeyn Muhâsebeciliği ihsân ve mukayyedü'l-ism-i defter-i hâcegân oldu."
    },
    {
      "subheading": "Zikr-i Mîrâhûr-ı Evvel-şüden-i Mehmed Efendi",
      "body": "Şehzâdelik vaktinde hıdmet-i taʻlîm ile şeref-yâb olan Mehmed Efendi hisse-mend-i nevâle-i ihsân-ı Şehin-şâhî olmak irâdesiyle Mîrâhûr-ı Evvel nasb u taʻyîn ve Kapucu-başılık ile kavâʻid-i binâ-yı emeli tarsîn olundu."
    },
    {
      "subheading": "Nasb-ı Kâtib-i Ağa-yı Dâru's-saʻâde",
      "body": "Kahveci-başı olan Mehmed Efendi emekdâr ve hıdmet-i sâbıkasına mücâzât-ı muktezâ-yı mürû’et bulunduğu bedîdâr olmağla, şehr-i Receb'in on üçüncü"
    }
  ]
}


{
  "page": 148,
  "events": [
    {
      "subheading": null,
      "body": "Pençşenbih günü matmah-ı nazarı olan Yazıcılık ile şemʻ-i bahtı fîrûzân ve husûl-i merâmıyla mazhar-ı birr ü ihsân oldu."
    },
    {
      "subheading": "Itlâk-ı Şeyhulislâm-ı esbâk İbrâhîm Bey ve mahdûmeş ve ʻafv-ı Ser-etıbbâ-yı Hâssa-i esbak",
      "body": "Şeyhulislâm-ı esbak ‘İvaz Mehmed Paşa-zâde İbrâhîm Beyefendi bundan akdem sevk-i kazâ vü kader ile Ankara'ya iclâ olunup, hevasıyla imtizâc edemediğinden, Burusa'ya nakl olunmasiyçün dâmen-gîr-i ilticâ ve hakkında sûret-i müsâʻade rû-nümâ olmuşidi. Kezâlik mahdûmları Mustafa Beyefendi sâhil-hânesinde me’mûren ikāmet ile tertîb-sâz-ı bezm-i sûz u güdâz ve sâbıkā Hekîm-başı Hayrullah Efendi Tâyif'de temekkün etmek üzere muhtâr-ı tarîk-i Hicâz olduğu sâmiʻa-res-i Şehriyâr-ı bende-nüvâz oldukda, üçünün birden ıtlâkına emr-i hümâyûn sâdır olup, Hayrullah Efendi'nin ıtlâk ve Âsitane'ye iltihâk fermânı pâ-der-rikâb-ı ʻazîmet olan Surre [9a] Emîni Çelebi Mehmed Efendi'ye iʻtâ ve Burusa'dan Âsitâne'ye gelmek üzere Şeyhulislâm-ı esbak müşârun ileyhe bir kıtʻa emr-i kazâ-mazâ isrâ ve mahdûmları dahi haber-i ʻavf ile tefrîh ve tercîh etdiği mahalde ikāmeti tasrîh olundu."
    },
    {
      "subheading": "ʻAzîmet-kerden-i Şehriyâr-ı diyânet-âsâr be-edâ-yı salât-ı Cumʻa bîş ez tekallüd-i seyf",
      "body": "Ezmân-ı sâbıkada mülûk ü selâtîn-i ʻizâm hazerâtı tekallüd-i seyf ʻakabinde edâ-yı salât-ı Cumʻa'yı iʻtiyâd ve kable't-tekallüd debdebe-i Mülûkâne ile bir mahalle ʻazîmeti istibʻâd ederler idi. Taklîd-i seyf olunmazdan mukaddem yevm-i mübârek-i Cumʻa kudûm edüp, kâffe-i nâs te’hîr-i salât-ı Cumʻa'da yek-sâk-ı vifâk iken Pâdişâh-ı takvâ-penâh e‘azzellahu ve kuvvahü hazretleri emr-i Hudâvend-i lâ-yezâlî âʻrâs-ı zâyileye takdîm ve mahzâ şeʻâyir-i İslâmiyye'yi taʻzîm niyyet-i hâlisesiyle Ayasofya"
    }
  ]
}
```

So, as we have seen in the text_extraction notebook, if there is no subheading, the event is continuing from one page to the next. This means that we need to design a loop that extends the text of one event if the first event in the subsequent JSON does not have a subheading.

In [115]:
def merge_continuations(clean_dir):
    """
    Load cleaned pages, merge cross-page continuations, track year markers.
    Returns a list of merged event dicts with keys:
        subheading, body, pages, year
    Year marker events are NOT included as standalone events in the output —
    they are recorded and stamped onto subsequent events.
    """
    clean_dir = Path(clean_dir)
    page_files = sorted(
        clean_dir.glob("page_*.json"),
        key=lambda p: int(p.stem.split("_")[1])
    )

    merged_events = []
    current_event = None
    current_year  = 1203   # carries forward until next year marker
    # i am using the hijri year because this is the first year covered in this book.
    # going forward, this will need to be updated manually for each new book, hence it should go on the metadata

    for pf in page_files:
        with open(pf, "r", encoding="utf-8") as f:
            page = json.load(f)

        page_num = page.get("page")
        events   = page.get("events", [])

        for ev in events:
            if not isinstance(ev, dict):
                continue

            sub  = ev.get("subheading")
            body = ev.get("body") or ""

            # ── Year marker ───────────────────────────────────
            # Subheading matches VEKĀYİ pattern and body is null/empty
            if sub and VEKAYI_PATTERN.search(sub) and not body.strip():
                # Flush current event first
                if current_event is not None:
                    merged_events.append(current_event)
                    current_event = None
                current_year += 1
                continue  # year markers don't become events

            # ── Continuation (no subheading) ──────────────────
            if sub is None:
                if current_event is not None:
                    if body:
                        prev_body = current_event.get("body") or ""
                        current_event["body"] = f"{prev_body}\n{body}".strip()
                    current_event.setdefault("pages", []).append(page_num)
                else:
                    # Edge case: very first event in the corpus has no subheading
                    current_event = {
                        "subheading": None,
                        "body":       body,
                        "pages":      [page_num],
                        "year":       current_year
                    }
                continue

            # ── New event ─────────────────────────────────────
            if current_event is not None:
                merged_events.append(current_event)

            current_event = {
                "subheading": sub,
                "body":       body,
                "pages":      [page_num],
                "year":       current_year
            }

    # Flush last event
    if current_event is not None:
        merged_events.append(current_event)

    return merged_events

In [116]:
merged_events = merge_continuations(CLEAN_DIR)

print(f"Total merged events: {len(merged_events)}")

# Quick sanity check: how many have a year assigned?
with_year    = sum(1 for e in merged_events if e.get("year"))
#without_year = sum(1 for e in merged_events if not e.get("year"))
print(f"  With year:    {with_year}")
#print(f"  Without year: {without_year}  ← expected for events before first VEKĀYİ marker")

# Preview first 5
print("\n" + "=" * 80)
for i, ev in enumerate(merged_events[:5], start=1):
    print(f"Event {i}")
    print(f"  Year:      {ev.get('year')}")
    print(f"  Pages:     {ev.get('pages')}")
    print(f"  Subheading: {ev.get('subheading')}")
    print(f"  Body:      {(ev.get('body') or '')[:150]}...")
    print("-" * 80)

Total merged events: 389
  With year:    389

Event 1
  Year:      1203
  Pages:     [134, 135, 136, 136, 137, 138, 139, 140, 141, 142]
  Subheading: [1b] Zeyl-i Vâsıf li-Vâsıf Târîh-i Mehâsinü'l-Âsâr ve Hakāîkü'l-Ahbâr Bismillâhirrahmânirrahîm
  Body:      [Nazm:] Ey nigârende-i zemîn ü zemân, 
V'ey nesak-sâz-ı ‘âlem-i imkân! 
Safha-zîb-i sutûr olan eserim, 
Eyle makbûl, Pâdişâh-ı cihân. 

ʻAzamet ü kibr...
--------------------------------------------------------------------------------
Event 2
  Year:      1203
  Pages:     [143, 144, 144, 145, 145, 145, 146]
  Subheading: Zikr-i cülûs-ı meymenet-me’nûs-ı hazret-i Sultân Selîm Hân bin es-Sultân Mustafâ Hân bin es-Sultân Ahmed Hân
  Body:      Çünki nevbet-i Hilâfet-i ʻuzmâ ve ʻatıyye-i Saltanat-ı ʻulyâ bi'l-irsi ve'l istihkāk zübde-i Şehriyârân-ı âfâk ve güzîde-i Tâcdârân-ı ʻale'l-ıtlâk ola...
--------------------------------------------------------------------------------
Event 3
  Year:      1203
  Pages:     [146]
  Subheading: Ke

In [117]:
with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["event_id", "subheading", "body", "pages", "year"]
    )
    writer.writeheader()

    for i, ev in enumerate(merged_events, start=1):
        writer.writerow({
            "event_id":  i,
            "subheading": ev.get("subheading") or "",
            "body":       ev.get("body") or "",
            "pages":      ",".join(map(str, ev.get("pages", []))),
            "year":       ev.get("year") or ""
        })

print(f"✅ Saved {len(merged_events)} events to {CSV_FILE}")

✅ Saved 389 events to events.csv


## Comparing with the ToC

So far it all seems pretty good. 388 events seem like a reasonable number and the events by and large look clean. Before we conclude with this example and turn these notebooks into scripts that we can use for the rest of the works, let's check our work against the table of contents.

For this next step, I extracted the ToC and formatted it in a similar way to the events. There are some typos in the subheadings especially with the ʻ character so we will normalize before comparing the subheadings.

Using the ToC for comparison is not always the most reliable because the ToC can be wrong too. For instance, in this work, two subheadings on page 446 were missing and I only discovered it after running the comparison. I had to add them myself.

In [4]:
def turkish_lower(text):
    text = text.replace('I', 'ı').replace('İ', 'i')
    return text.lower()

In [5]:
def remove_special_chars(text):
    text = text.replace('â', 'a')
    text = text.replace('î', 'i')
    text = text.replace('û', 'u')
    text = text.replace('’', "")
    text = text.replace('ʿ', "")
    text = text.replace('ʾ', "")
    text = text.replace('ā', 'a')
    text = text.replace('ī', 'i')
    text = text.replace('ū', 'u')
    text = text.replace('ʻ', "")
    return text

In [6]:
def remove_manuscript_page(text):
    # Removes patterns like [1b] or [128b]
    return re.sub(r'\[\d+b\]', '', text)

In [7]:
import pandas as pd

In [8]:
with open(CSV_FILE, "r", encoding="utf-8") as f:
    gemini_df = pd.read_csv(f)

In [9]:
gemini_df.head()

,event_id,subheading,body,pages,year
0,1,[1b] Zeyl-i Vâsıf li-Vâsıf Târîh-i Mehâsinü'l-...,"[Nazm:] Ey nigârende-i zemîn ü zemân, \nV'ey n...","134,135,136,136,137,138,139,140,141,142",1203
1,2,Zikr-i cülûs-ı meymenet-me’nûs-ı hazret-i Sult...,Çünki nevbet-i Hilâfet-i ʻuzmâ ve ʻatıyye-i Sa...,"143,144,144,145,145,145,146",1203
2,3,Kethudâ-yı Bevvâbîn-şüden-i Şemseddîn Bey,Melek Ahmed Paşa-zâde Şemseddîn Bey'in sâbıka-...,146,1203
3,4,Firistâden-i mühr-i cedîd-i Şehriyâr-ı Cem-haş...,Ahbâr-ı sârre-i cülûs-ı meymenet-me’nûs bundan...,146,1203
4,5,Nasb-ı Kethudâ be-cânib-i Mehd-i ‘ulyâ,"Havvâ-menzilet, Belkīs-rifʻat, ʻAzrâ-tahâret, ...",146,1203


In [10]:
with open("vasif_1789-1794_toc.csv", "r", encoding="utf-8") as f:
    toc_df = pd.read_csv(f)

In [11]:
len(toc_df)

382

In [12]:
toc_df.head()

,subheading,page,year
0,[introductory materials],134,1203
1,Zikr-i cülûs-ı meymenet-me'nûs-ı hazret-i Sult...,143,1203
2,Kethudâ-yı Bevvâbîn-şüden-i Şemseddîn Bey,146,1203
3,Firistâden-i mühr-i cedîd-i Şehriyâr-ı Cem-haş...,146,1203
4,Nasb-ı Kethudâ be-cânib-i Mehd-i ʻulyâ,146,1203


In [13]:
gemini_df["subheading_clean"] = gemini_df["subheading"].apply(remove_special_chars).apply(turkish_lower).apply(remove_manuscript_page)

In [14]:
toc_df["subheading_clean"] = toc_df["subheading"].apply(remove_special_chars).apply(turkish_lower)

In [15]:
gemini_subheadings = list(gemini_df["subheading_clean"].dropna())

In [16]:
toc_subheadings = list(toc_df["subheading_clean"].dropna())

In [17]:
toc_subheadings[:5]

['[introductory materials]',
 "zikr-i cülus-ı meymenet-me'nus-ı hazret-i sultan selim han bin es-sultan mustafa han bin es-sultan ahmed han",
 'kethuda-yı bevvabin-şüden-i şemseddin bey',
 'firistaden-i mühr-i cedid-i şehriyar-ı cem-haşem be-canib-i sadrıazam',
 'nasb-ı kethuda be-canib-i mehd-i ulya']

In [18]:
gemini_subheadings[:5]

[" zeyl-i vasıf li-vasıf tarih-i mehasinü'l-âsar ve hakaikü'l-ahbar bismillahirrahmanirrahim",
 'zikr-i cülus-ı meymenet-menus-ı hazret-i sultan selim han bin es-sultan mustafa han bin es-sultan ahmed han',
 'kethuda-yı bevvabin-şüden-i şemseddin bey',
 'firistaden-i mühr-i cedid-i şehriyar-ı cem-haşem be-canib-i sadrıazam',
 'nasb-ı kethuda be-canib-i mehd-i ‘ulya']

In [19]:
import difflib

In [20]:
def find_missing_entries(baseline, ocr_to_check, threshold=0.8):
    """
    Compares baseline against ocr_to_check.
    Returns entries from baseline that don't have a fuzzy match in ocr_to_check.
    """
    missing = []
    
    for item in baseline:
        # get_close_matches looks for the best matches in the second list
        matches = difflib.get_close_matches(item, ocr_to_check, n=1, cutoff=threshold)
        
        if not matches:
            missing.append(item)
            
    return missing

# Run the check
missing_items = find_missing_entries(gemini_subheadings, toc_subheadings)

print(f"--- Found {len(missing_items)} missing or significantly different entries ---")
for item in missing_items:
    print(f"MISSING: {item}")

--- Found 8 missing or significantly different entries ---
MISSING:  zeyl-i vasıf li-vasıf tarih-i mehasinü'l-âsar ve hakaikü'l-ahbar bismillahirrahmanirrahim
MISSING: tafsili bu ki,
MISSING: tarih-i veladet-i hazret-i şehriyari;
MISSING: müşarun ileyhin işar-ı ab-darındandır;
MISSING: biz yine sadede gelelim. serasker paşa'nın keyfiyyeti ve ordunun vuku-ı haleti ru'esa-yı ‘asakir ve kelanteran-ı kabayil taraflarından ber-vech-i işba maruz-ı 'atebe-i gerdun ittisa kılınup, ahar serasker talebiyle telafi-yi ma-fat kaydında oldukları müteayyin oldukda
MISSING: battal-zade nuri mehmed paşa
MISSING: filibe kasabası'nda merfuu'l-vezare mukim olan mikdad ahmed paşa ve canik taraflarında olan battal paşa-zade hayreddin paşa'nın izale-i vücudları
MISSING: mora valisi olan cezayirli kethudası vezir ‘ali paşa'nın zulm ü teaddisi


exists in gemini_subheadings but not in toc_subheadings (ie subheadings that are added during the text extraction)

- MISSING:  zeyl-i vasıf li-vasıf tarih-i mehasinü'l-âsar ve hakaikü'l-ahbar bismillahirrahmanirrahim -> this is the first entry, ignore
- MISSING: tafsili bu ki, -> we should add ending in , as an issue to be fixed
- MISSING: tarih-i veladet-i hazret-i şehriyari; -> ; also issue
- MISSING: müşarun ileyhin işar-ı ab-darındandır; -> same
  
these need further investigation
- MISSING: biz yine sadede gelelim. serasker paşa'nın keyfiyyeti ve ordunun vuku-ı haleti ru'esa-yı ‘asakir ve kelanteran-ı kabayil taraflarından ber-vech-i işba maruz-ı 'atebe-i gerdun ittisa kılınup, ahar serasker talebiyle telafi-yi ma-fat kaydında oldukları müteayyin oldukda
- MISSING: battal-zade nuri mehmed paşa
- MISSING: filibe kasabası'nda merfuu'l-vezare mukim olan mikdad ahmed paşa ve canik taraflarında olan battal paşa-zade hayreddin paşa'nın izale-i vücudları
- MISSING: mora valisi olan cezayirli kethudası vezir ‘ali paşa'nın zulm ü teaddisi

In [21]:
# Run the check
missing_items_alt = find_missing_entries(toc_subheadings, gemini_subheadings)

print(f"--- Found {len(missing_items_alt)} missing or significantly different entries ---")
for item in missing_items_alt:
    print(f"MISSING: {item}")

--- Found 1 missing or significantly different entries ---
MISSING: [introductory materials]


exists in toc_subheadings but not in gemini_subheadings, ie these are missing from the text extraction or we accidentally merged them in our clean up

- MISSING: [introductory materials] -> this is the first entry, which did not exist in the toc because it does not have an official title so i added this as a placeholder

In [22]:
def align_ocr_lists(baseline, comparison, threshold=0.8):
    alignment_data = []

    for item_a in baseline:
        # Find the best single match in list_b
        matches = difflib.get_close_matches(item_a, comparison, n=1, cutoff=threshold)
        
        if matches:
            item_b = matches[0]
            # Calculate exact similarity ratio (0.0 to 1.0)
            score = difflib.SequenceMatcher(None, item_a, item_b).ratio()
        else:
            item_b = "--- MISSING/NO MATCH ---"
            score = 0.0
            
        alignment_data.append({
            "Correct_List_A": item_a,
            "Matched_List_B": item_b,
            "Similarity": round(score, 2)
        })

    return pd.DataFrame(alignment_data)

# Create the DataFrame
df = align_ocr_lists(gemini_subheadings, toc_subheadings)

# Display settings for better visibility in console
pd.set_option('display.max_colwidth', 50)
print(df)

                                        Correct_List_A  \
0     zeyl-i vasıf li-vasıf tarih-i mehasinü'l-âsar...   
1    zikr-i cülus-ı meymenet-menus-ı hazret-i sulta...   
2            kethuda-yı bevvabin-şüden-i şemseddin bey   
3    firistaden-i mühr-i cedid-i şehriyar-ı cem-haş...   
4               nasb-ı kethuda be-canib-i mehd-i ‘ulya   
..                                                 ...   
384          meks-i kapudan-ı derya vezir hüseyin paşa   
385                              ‘azl-i sadr-ı anadolu   
386  ihsan-ı paye-i rumeli be-dağıstani ibrahim efendi   
387                                             tezyil   
388                                             hikmet   

                                        Matched_List_B  Similarity  
0                             --- MISSING/NO MATCH ---        0.00  
1    zikr-i cülus-ı meymenet-me'nus-ı hazret-i sult...        1.00  
2            kethuda-yı bevvabin-şüden-i şemseddin bey        1.00  
3    firistaden-i mühr-i ce

In [23]:
# Create the DataFrame
df_alt = align_ocr_lists(toc_subheadings, gemini_subheadings)

# Display settings for better visibility in console
pd.set_option('display.max_colwidth', 50)
print(df_alt)

                                        Correct_List_A  \
0                             [introductory materials]   
1    zikr-i cülus-ı meymenet-me'nus-ı hazret-i sult...   
2            kethuda-yı bevvabin-şüden-i şemseddin bey   
3    firistaden-i mühr-i cedid-i şehriyar-ı cem-haş...   
4                nasb-ı kethuda be-canib-i mehd-i ulya   
..                                                 ...   
377          meks-i kapudan-ı derya vezir hüseyin paşa   
378                               azl-i sadr-ı anadolu   
379  ihsan-ı paye-i rumeli be-dağıstani ibrahim efendi   
380                                             tezyil   
381                                             hikmet   

                                        Matched_List_B  Similarity  
0                             --- MISSING/NO MATCH ---        0.00  
1    zikr-i cülus-ı meymenet-menus-ı hazret-i sulta...        1.00  
2            kethuda-yı bevvabin-şüden-i şemseddin bey        1.00  
3    firistaden-i mühr-i ce

In [24]:
# let's merge these back with their original dfs

# 1. Reset index on both to be absolutely sure they align by row position
toc_df = toc_df.reset_index(drop=True)
df_alt = df_alt.reset_index(drop=True)

# 2. Merge using the index instead of the text column
# This ensures row 377 in toc_df only talks to row 377 in match_df
toc_final = toc_df.merge(
    df_alt[['Matched_List_B', 'Similarity']], 
    left_index=True, 
    right_index=True, 
    how='left'
)

# 3. Validation: The row count should now be identical to your original toc_df
print(f"Original row count: {len(toc_df)}")
print(f"Final row count: {len(toc_final)}")

Original row count: 382
Final row count: 382


In [25]:
toc_final.head()

,subheading,page,year,subheading_clean,Matched_List_B,Similarity
0,[introductory materials],134,1203,[introductory materials],--- MISSING/NO MATCH ---,0.00
1,Zikr-i cülûs-ı meymenet-me'nûs-ı hazret-i Sult...,143,1203,zikr-i cülus-ı meymenet-me'nus-ı hazret-i sult...,zikr-i cülus-ı meymenet-menus-ı hazret-i sulta...,1.00
2,Kethudâ-yı Bevvâbîn-şüden-i Şemseddîn Bey,146,1203,kethuda-yı bevvabin-şüden-i şemseddin bey,kethuda-yı bevvabin-şüden-i şemseddin bey,1.00
3,Firistâden-i mühr-i cedîd-i Şehriyâr-ı Cem-haş...,146,1203,firistaden-i mühr-i cedid-i şehriyar-ı cem-haş...,firistaden-i mühr-i cedid-i şehriyar-ı cem-haş...,1.00
4,Nasb-ı Kethudâ be-cânib-i Mehd-i ʻulyâ,146,1203,nasb-ı kethuda be-canib-i mehd-i ulya,nasb-ı kethuda be-canib-i mehd-i ‘ulya,0.99


In [26]:
gemini_df = gemini_df.reset_index(drop=True)
df = df.reset_index(drop=True)

gemini_final = gemini_df.merge(
    df[['Matched_List_B', 'Similarity']], 
    left_index=True, 
    right_index=True, 
    how='left'
)

print(f"Original row count: {len(gemini_df)}")
print(f"Final row count: {len(gemini_final)}")


Original row count: 389
Final row count: 389


In [27]:
gemini_final.head()

,event_id,subheading,body,pages,year,subheading_clean,Matched_List_B,Similarity
0,1,[1b] Zeyl-i Vâsıf li-Vâsıf Târîh-i Mehâsinü'l-...,"[Nazm:] Ey nigârende-i zemîn ü zemân, \nV'ey n...","134,135,136,136,137,138,139,140,141,142",1203,zeyl-i vasıf li-vasıf tarih-i mehasinü'l-âsar...,--- MISSING/NO MATCH ---,0.00
1,2,Zikr-i cülûs-ı meymenet-me’nûs-ı hazret-i Sult...,Çünki nevbet-i Hilâfet-i ʻuzmâ ve ʻatıyye-i Sa...,"143,144,144,145,145,145,146",1203,zikr-i cülus-ı meymenet-menus-ı hazret-i sulta...,zikr-i cülus-ı meymenet-me'nus-ı hazret-i sult...,1.00
2,3,Kethudâ-yı Bevvâbîn-şüden-i Şemseddîn Bey,Melek Ahmed Paşa-zâde Şemseddîn Bey'in sâbıka-...,146,1203,kethuda-yı bevvabin-şüden-i şemseddin bey,kethuda-yı bevvabin-şüden-i şemseddin bey,1.00
3,4,Firistâden-i mühr-i cedîd-i Şehriyâr-ı Cem-haş...,Ahbâr-ı sârre-i cülûs-ı meymenet-me’nûs bundan...,146,1203,firistaden-i mühr-i cedid-i şehriyar-ı cem-haş...,firistaden-i mühr-i cedid-i şehriyar-ı cem-haş...,1.00
4,5,Nasb-ı Kethudâ be-cânib-i Mehd-i ‘ulyâ,"Havvâ-menzilet, Belkīs-rifʻat, ʻAzrâ-tahâret, ...",146,1203,nasb-ı kethuda be-canib-i mehd-i ‘ulya,nasb-ı kethuda be-canib-i mehd-i ulya,0.99


We need to be more refined about this search because some titles like Garibe or Tezyil appear multiple times so if they are missing in a specific place, they don't get flagged.

In [28]:
def windowed_fuzzy_match(source_list, target_list, window_size=20, cutoff=0.8):
    """
    Compares source_list to target_list, but only searches for a match
    within a 'window' of indices around the current position.
    """
    results = []
    
    for i, item in enumerate(source_list):
        # Define search window in target list (e.g., current index +/- 25)
        start = max(0, i - window_size)
        end = min(len(target_list), i + window_size)
        search_area = target_list[start:end]
        
        # Find the best match in the restricted neighborhood
        matches = difflib.get_close_matches(str(item), search_area, n=1, cutoff=cutoff)
        
        if matches:
            match_val = matches[0]
            score = difflib.SequenceMatcher(None, str(item), match_val).ratio()
            status = "Matched"
        else:
            match_val = None
            score = 0.0
            status = "MISSING"
            
        results.append({
            'original_index': i,
            'subheading': item,
            'match_found': match_val,
            'similarity': round(score, 3),
            'status': status
        })
    return pd.DataFrame(results)

# --- RUNNING THE CHECK ---
# 1. To see what the TOC is missing (Check Gemini vs TOC)
missing_in_toc = windowed_fuzzy_match(gemini_subheadings, toc_subheadings)

# 2. To see what the OCR missed (Check TOC vs Gemini)
missing_in_ocr = windowed_fuzzy_match(toc_subheadings, gemini_subheadings)

# Filter to see only the discrepancies
gaps = missing_in_toc[missing_in_toc['status'] == "MISSING"]
print(gaps[['original_index', 'subheading']])

     original_index                                         subheading
0                 0   zeyl-i vasıf li-vasıf tarih-i mehasinü'l-âsar...
66               66                                     tafsili bu ki,
91               91              tarih-i veladet-i hazret-i şehriyari;
92               92             müşarun ileyhin işar-ı ab-darındandır;
164             164  biz yine sadede gelelim. serasker paşa'nın key...
198             198                       battal-zade nuri mehmed paşa
199             199  filibe kasabası'nda merfuu'l-vezare mukim olan...
200             200  mora valisi olan cezayirli kethudası vezir ‘al...


In [29]:
gaps_alt = missing_in_ocr[missing_in_ocr['status'] == "MISSING"]
print(gaps_alt[['original_index', 'subheading']])

   original_index                subheading
0               0  [introductory materials]


Perfect! so it seems like all the issues that we have are because of the titles in OCR that are not in the ToC. These are the ones that were mistakenly extracted as subheadings, so we will merge them up. instead of editing the transliteration_json, let's fix this in the csv itself.

These are our problem entries

66               66                                     tafsili bu ki,

91               91              tarih-i veladet-i hazret-i şehriyari;

92               92             müşarun ileyhin işar-ı ab-darındandır;

164             164  biz yine sadede gelelim. serasker paşa'nın key...

198             198                       battal-zade nuri mehmed paşa

199             199  filibe kasabası'nda merfuu'l-vezare mukim olan...

200             200  mora valisi olan cezayirli kethudası vezir ‘al...

Let's first print how they look now and what we want to do next

In [30]:
import textwrap

def preview_merges(df, to_fix_list):
    """
    Prints the full text of proposed merges using bottom-up logic.
    Shows exactly how text will be nested before the actual transformation.
    """
    # Sort descending to reflect the bottom-up nesting logic
    to_fix_list = sorted(list(set(to_fix_list)), reverse=True)
    
    print(f"{'#'*100}")
    print(f"{'FULL TEXT NESTED PREVIEW (BOTTOM-UP)':^100}")
    print(f"{'#'*100}\n")

    for idx in to_fix_list:
        if idx == 0: continue
        
        # In a nested bottom-up approach, the immediate target is always the row above
        target_idx = idx - 1
            
        print(f"👉 PROPOSAL: Merge Row {idx} UP into Row {target_idx}")
        print(f"   (If {target_idx} is also in the fix list, this will later merge further up.)")
        print("-" * 30)
        
        # Print Target (Anchor for this specific step)
        print(f"  [TARGET] Index: {target_idx}")
        print(f"    Subheading: {df.at[target_idx, 'subheading']}")
        # Wrap and truncate body for legibility
        wrapped_target = textwrap.fill(df.at[target_idx, 'body'], width=90, initial_indent=' Text: ', subsequent_indent=' ')
        #print(f"    Full Body:  {df.at[target_idx, 'body']}")
        if len(wrapped_target) > 400:
            print(wrapped_target[:350] + "\n [...] " + wrapped_target[-200:])
        else:
            print(wrapped_target)
        print("\n")
        
        # Print Fragment
        print(f"  [FRAGMENT] Index: {idx}")
        print(f"    Subheading: {df.at[idx, 'subheading']}")
        #print(f"    Full Body:  {df.at[idx, 'body']}")
        wrapped_fragment = textwrap.fill(df.at[idx, 'body'], width=90, initial_indent=' Text: ', subsequent_indent=' ')
        #print(f"    Full Body:  {df.at[idx, 'body']}")
        if len(wrapped_fragment) > 400:
            print(wrapped_fragment[:350] + "\n [...] " + wrapped_fragment[-200:])
        else:
            print(wrapped_fragment)
        print("\n")
        print(f"\n{'='*100}\n")


In [31]:
to_fix = [66, 91, 92, 164, 198, 199, 200]
preview_merges(gemini_final, to_fix)

####################################################################################################
                                FULL TEXT NESTED PREVIEW (BOTTOM-UP)                                
####################################################################################################

👉 PROPOSAL: Merge Row 200 UP into Row 199
   (If 199 is also in the fix list, this will later merge further up.)
------------------------------
  [TARGET] Index: 199
    Subheading: Filibe Kasabası'nda merfûʻu'l-Vezâre mukīm olan Mikdâd Ahmed Paşa ve Canik taraflarında olan Battâl Paşa-zâde Hayreddin Paşa'nın izâle-i vücûdları
 Text: muktezayât-ı vakt ü hâlden idügi ‘atebe-i ‘ulyâya inhâ olunduğuna binâ’en ikisi
 dahi i‘dâm ve ser-i maktûʻları mevzû‘-ı siyaset-gâh-ı gerden-keşân-ı enâm oldu.


  [FRAGMENT] Index: 200
    Subheading: Mora Vâlîsi olan Cezâyirli Kethudâsı Vezîr ‘Ali Paşa'nın zulm ü teʻaddîsi
 Text: şöhret-gîr ve hakkında birkaç defa rikʻa-i iştikâ olunup, rahmen li'l-‘ibâd 

In [34]:
def perform_bottom_up_merge(df, indices_to_merge):
    """
    Performs merges from bottom to top to handle nested fragments.
    Each row in 'indices_to_merge' is appended to the row directly above it (idx - 1).
    """
    # Sort descending to handle nesting (e.g., 92 -> 91, 91 -> 90, 90 -> 89)
    sorted_indices = sorted(indices_to_merge, reverse=True)
    
    # Work on a copy of the dataframe
    working_df = df.copy()
    
    for idx in sorted_indices:
        target_idx = idx - 1
        
        # 2. Extract fragment subheading and body
        sub = working_df.at[idx, 'subheading']
        body = working_df.at[idx, 'body']
        
        # 3. Format the text for insertion (turning what was flagged as a subheading into a new paragraph and merging it with the body with a space in between)
        formatted_sub = f"\n\n{sub} " if pd.notna(sub) and str(sub).strip() != "" else "\n\n"
        content_to_add = f"{formatted_sub}{body}"
        
        # 4. Append the text to the target row above
        working_df.at[target_idx, 'body'] = f"{working_df.at[target_idx, 'body']}{content_to_add}"
        
        # 5. Drop the row that was merged up
        working_df = working_df.drop(index=idx)
        
    # 6. Reset the index once all transformations are complete
    final_df = working_df.reset_index(drop=True)
    # 7. RESET THE EVENT_ID COLUMN
    # This ensures your data column is also sequential (1, 2, 3...)
    final_df['event_id'] = final_df.index + 1
    return final_df

In [35]:
final_cleaned_df = perform_bottom_up_merge(gemini_final, to_fix)

final_cleaned_df.drop(columns=['subheading_clean', 'Matched_List_B', 'Similarity'], inplace=True, errors='ignore')

# Save the final cleaned version
final_cleaned_df.to_csv("events_final.csv", index=False)

print(f"Original row count: {len(gemini_final)}")
print(f"Final row count:   {len(final_cleaned_df)}")

Original row count: 389
Final row count:   382
